In [1]:
FILE_PATH  = "/content/Copy of Ardook_Data_All_Grant_Recommendations_UPDATED_v3 (2).xlsx"   # path after uploading via the Files pane
SHEET_NAME = "Grant Tracker"
TEXT_COL   = "Core Requirements"             # column being compared to the profile
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

In [2]:
import pandas as pd
import numpy as np

df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
df.columns = df.columns.str.strip()          # guards against stray spaces in headers
print(df.shape)                              # expect (37, 13)
print("Empty requirement cells:", df[TEXT_COL].isna().sum())

# Keep an ID, the text we score, and Fit (ground truth: carried along, never scored)
data = df[["Priority / Deadline Order", "Program", TEXT_COL, "Fit"]].copy()
data[TEXT_COL] = data[TEXT_COL].fillna("").astype(str).str.strip()
data.head()

(37, 13)
Empty requirement cells: 0


,Priority / Deadline Order,Program,Core Requirements,Fit
0,1,Women TechEU 2026–2028,Woman founder/co-founder in top management; wo...,Excellent
1,2,Tara’s Circle 2026,Women-led; international; seed-stage; defined/...,Excellent
2,3,Her Agenda Breakthrough Grant,Woman entrepreneur; operating business; newsle...,Strong
3,4,Hey Helen Grant (Visionaries),Woman; ≥50% ownership; U.S. resident; U.S. for...,Strong
4,5,WomensNet Amber Grant / Startup Grant,Women-owned business; Startup Grant track for ...,Strong


In [3]:
PROFILE_SENTENCES = [
    "Woman-founded and woman-led company.",
    "For-profit company.",
    "Early-stage, pre-revenue startup.",
    "Builds AI and data technology tools.",
    "Focus on women's economic empowerment.",
    "Based in the United States.",            # edit to match Ardook
]
PROFILE_TEXT = " ".join(PROFILE_SENTENCES)

In [4]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(MODEL_NAME)

def score_coarse(texts, profile_text, model):
    """Whole requirement string vs. whole profile -> one cosine score per grant."""
    p = model.encode([profile_text], normalize_embeddings=True)   # shape (1, 384)
    g = model.encode(list(texts),   normalize_embeddings=True)    # shape (n, 384)
    return (g @ p.T).ravel()     # normalized vectors, so dot product = cosine; (n,1) -> (n,)

def score_granular(texts, profile_sents, model, agg=np.mean):
    """Split requirements on ';'. For each criterion, take its best-matching profile
    sentence (max), then aggregate across criteria (mean by default; try np.min)."""
    P = model.encode(profile_sents, normalize_embeddings=True)    # (k, 384)
    scores = []
    for t in texts:
        criteria = [c.strip() for c in t.split(";") if c.strip()]
        if not criteria:
            scores.append(np.nan)
            continue
        C = model.encode(criteria, normalize_embeddings=True)      # (m, 384)
        sims = C @ P.T                                             # (m, k): criterion x profile sentence
        scores.append(agg(sims.max(axis=1)))                       # best match per criterion, then aggregate
    return np.array(scores)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
data["score_coarse"]   = score_coarse(data[TEXT_COL], PROFILE_TEXT, model)
data["score_granular"] = score_granular(data[TEXT_COL], PROFILE_SENTENCES, model)

ranked = data.sort_values("score_coarse", ascending=False).reset_index(drop=True)
ranked.insert(0, "rank", ranked.index + 1)

pd.set_option("display.max_colwidth", 80)
ranked[["rank", "Program", "Fit", "score_coarse", "score_granular", TEXT_COL]]

,rank,Program,Fit,score_coarse,score_granular,Core Requirements
0,1,Oasis Collective × AI After 5 – Women in AI Pitch Competition (SF),Excellent,0.709520,0.420341,Women-led AI startup; investor-ready orientation; SF competition
1,2,Cartier Women’s Initiative – Science & Technology Pioneer Award,Strong if deep-tech criteria met,0.654203,0.461106,Women-led science/technology venture; edition-specific criteria
2,3,Giving Joy Grant,Moderate,0.638994,0.571030,"Women entrepreneurs; proposed activity must benefit women, girls and/or fami..."
3,4,Google for Startups Women Founders Fund,Excellent if a matching round opens,0.629406,0.423225,Woman founder; full-time founder; Pre-seed–Series A; scalable product/servic...
4,5,Women TechEU 2026–2028,Excellent,0.621485,0.445330,Woman founder/co-founder in top management; women collectively ≥25% ownershi...
5,6,EchoHer Founder Fellowship,Excellent,0.616589,0.479402,"Early-stage women/non-binary founders; selection on problem, solution, impac..."
6,7,She Believed She Could – Iconic Impact / Emerging Founder Grant,Strong,0.584855,0.402759,Female entrepreneur; growth potential; impact/leadership/service criteria
7,8,WomensNet Amber Grant / Startup Grant,Strong,0.578531,0.632324,Women-owned business; Startup Grant track for idea/minimal-sales businesses
8,9,DivInc Women in Tech Accelerator,Strong thematic fit; low practical fit,0.573963,0.370656,Women-led early-stage tech/software; pilot/beta/MVP pathway; major CEO time ...
9,10,Tara’s Circle 2026,Excellent,0.551843,0.376921,Women-led; international; seed-stage; defined/functional MVP; measurable soc...


In [6]:
print("Score range:", ranked["score_coarse"].min().round(3), "to", ranked["score_coarse"].max().round(3))
print(ranked[ranked["Program"].str.contains("GTIA")][["rank", "Program", "score_coarse"]])  # the nonprofit-only test case

AttributeError: 'float' object has no attribute 'round'

In [ ]:
out_name = f"ranked_{MODEL_NAME.split('/')[-1]}.csv"
ranked.to_csv(out_name, index=False)
# from google.colab import files; files.download(out_name)   # uncomment to download